In [1]:
import os
import sys
from collections import Counter
import pandas as pd

# 1. Βρίσκουμε το root του project (δύο επίπεδα πάνω από mba/Georgios)
root_dir = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.append(root_dir)

from config import *

print("Project root:", root_dir)
print("ORDERS_PATH:", ORDERS_PATH)
print("TRAIN_PATH:", ORDER_PRODUCTS__TRAIN_PATH)

# 2. Φορτώνουμε τα TRAIN δεδομένα από τα parquet του project
orders_df = pd.read_parquet(ORDERS_PATH)
op_train = pd.read_parquet(ORDER_PRODUCTS__TRAIN_PATH)

# Κρατάμε μόνο τα train orders (ασφάλεια)
train_order_ids = orders_df.loc[orders_df["eval_set"] == "train", "order_id"]
op_train = op_train[op_train["order_id"].isin(train_order_ids)]

# 3. Φτιάχνουμε baskets: set(product_id) ανά order_id
baskets_series = (
    op_train
    .groupby("order_id")["product_id"]
    .apply(set)
)

baskets = list(baskets_series)
print(f"Number of baskets (TRAIN): {len(baskets)}")

# 4. Μετράμε συχνότητες των single items
item_counts = Counter()
for basket in baskets:
    item_counts.update(basket)

num_unique_items = len(item_counts)
print(f"Unique items in TRAIN: {num_unique_items}")


Project root: c:\Users\Georg\Desktop\Computational_Tools
ORDERS_PATH: C:\Users\Georg\Desktop\Computational_Tools\data\1_cleaned\orders.pq
TRAIN_PATH: C:\Users\Georg\Desktop\Computational_Tools\data\1_cleaned\order_products__train.pq
Number of baskets (TRAIN): 2933665
Unique items in TRAIN: 49641


Count single items

In [2]:
support_percentage = 0.001     # 0.1%
support_threshold = int(len(baskets) * support_percentage)
support_threshold

2933

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Multipliers based on theory (1x, 1.5x, 2x, 3x)
multipliers = [1.0, 1.5, 2.0, 3.0]

# Support percentages for testing PCY Pass 1
support_percentages = [0.001, 0.002, 0.005, 0.01]  # 0.1%, 0.2%, 0.5%, 1%

# Calculate bucket sizes
bucket_options = [int(num_unique_items * m) for m in multipliers]
print("Bucket sizes:", bucket_options)

# Store results in a matrix: rows = multipliers, columns = support thresholds
results = np.zeros((len(multipliers), len(support_percentages)), dtype=int)

for i, num_buckets in enumerate(bucket_options):
    print(f"\n--- Testing bucket size {num_buckets} ---")
    
    for j, sp in enumerate(support_percentages):
        support_threshold = int(len(baskets) * sp)
        print(f"  Support {sp} → threshold {support_threshold}")
        
        bucket_counts = np.zeros(num_buckets, dtype=int)
        
        # Hashing pairs
        for basket in baskets:
            b = sorted(basket)
            for a in range(len(b)):
                for c in range(a + 1, len(b)):
                    h = (b[a] * 13 + b[c] * 7) % num_buckets
                    bucket_counts[h] += 1
        
        frequent_buckets = (bucket_counts >= support_threshold).sum()
        results[i, j] = frequent_buckets

print("\nResult matrix:\n")
print(results)

# Plot heatmap
plt.figure(figsize=(10, 6))
plt.imshow(results, cmap='viridis', aspect='auto')
plt.colorbar(label="Frequent Buckets")

plt.xticks(range(len(support_percentages)), [f"{p*100:.1f}%" for p in support_percentages])
plt.yticks(range(len(bucket_options)), bucket_options)

plt.xlabel("Support Threshold (%)")
plt.ylabel("Number of Buckets")
plt.title("Frequent Buckets Across Different Bucket Sizes and Support Thresholds")
plt.show()


Bucket sizes: [49641, 74461, 99282, 148923]

--- Testing bucket size 49641 ---
  Support 0.001 → threshold 2933


### Selection of Bucket Size and Support Threshold

The combination of a bucket size equal to approximately 1.5× the number of unique products (≈75,000 buckets) and a support threshold of 0.5% was chosen based on the PCY Pass 1 evaluation matrix.  
This configuration provides balanced filtering: smaller bucket sizes produced excessive collisions, while larger sizes led to overly sparse bitmaps.  
At 75k buckets and 0.5% support, the number of frequent buckets remained low enough to ensure efficient pruning, while still preserving meaningful candidate pairs for Pass 2.  

## PCY Pass 1 – Saved State (Checkpoint)

This notebook has completed the full PCY Pass 1 process, including:
- Loading and preprocessing the Instacart data  
- Creating baskets  
- Counting single items  
- Testing multiple bucket sizes  
- Testing multiple support thresholds  
- Running the full hashing procedure  
- Building the frequent-bucket matrix  

To avoid re-running the 38-minute computation, the following objects have been saved to `pcy_pass1_results.pkl`:

- `baskets`  
- `num_unique_items`  
- `bucket_options`  
- `support_percentages`  
- `results` (frequent bucket matrix)


In [ ]:
import pickle

save_dir = DATA_PREPROCESSED_DIR / "pcy"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "pcy_pass1_results.pkl")

save_data = {
    "baskets": baskets,
    "num_unique_items": num_unique_items,
    "bucket_options": bucket_options,
    "support_percentages": support_percentages,
    "results": results
}

with open(save_path, "wb") as f:
    pickle.dump(save_data, f)

print(f"Saved at: {save_path}")